In [1]:
#!pip install tensorflow

In [1]:
# =========================
# 1. Imports
# =========================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [2]:
# =========================
# 2. Parameters
# =========================
IMG_SIZE=(224,224)
BATCH_SIZE=32
NUM_CLASSES=6
EPOCHS_STAGE1=15
EPOCHS_STAGE2=10

In [15]:
# =========================
# 3. Load Splits
# =========================
df = pd.read_csv("../../data/splits.csv")

train_df=df[df.split=="train"]
val_df=df[df.split=="val"]
test_df=df[df.split=="test"]

print(train_df.shape,val_df.shape,test_df.shape)

(10500, 4) (2250, 4) (2250, 4)


In [20]:
# =========================
# 4. Dataset pipeline
# =========================
def load_image(path,label):
    
    img=tf.io.read_file(path)
    img=tf.image.decode_png(img,channels=3)
    img=tf.image.resize(img,IMG_SIZE)

    # MobileNet preprocessing
    img=preprocess_input(img)

    return img,label

def create_dataset(df,training=False):
    
    paths=df["filepath"].values
    # Normalize paths: remove backslashes and prepend relative path
    paths = ["../../" + p.replace("\\", "/") for p in paths]
    
    labels=df["class_idx"].values
    
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    
    if training:
        ds=ds.shuffle(2000,seed=SEED)

    ds=ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds=ds.batch(BATCH_SIZE)
    ds=ds.prefetch(tf.data.AUTOTUNE)
    
    return ds


train_ds=create_dataset(train_df,training=True)
val_ds=create_dataset(val_df)
test_ds=create_dataset(test_df)

In [21]:
# Debug: Check first few image paths
print("First 3 image paths (after normalization):")
for i, p in enumerate(train_df["filepath"].values[:3]):
    normalized = "../../" + p.replace("\\", "/")
    exists = os.path.exists(normalized)
    print(f"  {normalized} | exists: {exists}")

First 3 image paths (after normalization):
  ../../data/raw/images/aerosol_cans/default/Image_124.png | exists: False
  ../../data/raw/images/shoes/real_world/Image_182.png | exists: False
  ../../data/raw/images/disposable_plastic_cutlery/default/Image_220.png | exists: True


In [22]:
# =========================
# 5. Augmentation
# =========================
augment=tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1)
])

In [23]:
# =========================
# 6. Class weights
# =========================
weights=compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df.class_idx),
    y=train_df.class_idx
)

class_weights={
    i:w for i,w in enumerate(weights)
}

print(class_weights)

{0: np.float64(0.38461538461538464), 1: np.float64(0.8333333333333334), 2: np.float64(2.5), 3: np.float64(1.25), 4: np.float64(1.6666666666666667), 5: np.float64(2.5)}


In [24]:
# =========================
# 7. Build model
# =========================
base_model=MobileNetV3Large(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable=False


inputs=tf.keras.Input(shape=(224,224,3))

x=augment(inputs)
x=base_model(x,training=False)

x=layers.GlobalAveragePooling2D()(x)
x=layers.BatchNormalization()(x)

x=layers.Dense(
    256,
    activation="relu"
)(x)

x=layers.Dropout(0.4)(x)

outputs=layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model=tf.keras.Model(inputs,outputs)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       246,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,247,750 (12.39 MB)

 Trainable params: 249,478 (974.52 KB)

 Non-trainable params: 2,998,272 (11.44 MB)

In [25]:
# =========================
# 8. Compile
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [26]:
# =========================
# 9. Callbacks
# =========================
callbacks=[
    EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        factor=0.3,
        patience=2
    ),

    ModelCheckpoint(
        "best_mobilenet.keras",
        save_best_only=True
    )
]

In [27]:
# =========================
# 10. Initial training
# =========================
history1=model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15


NotFoundError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
NewRandomAccessFile failed to Create/Open: ../../data/raw/images/plastic_trash_bags/default/Image_123.png : The system cannot find the path specified.
; No such process
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_23263]

In [ ]:
# =========================
# 11. Fine tuning
# unfreeze top layers only
# =========================
base_model.trainable=True

for layer in base_model.layers[:-40]:
    layer.trainable=False


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2=model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# =========================
# 12. Evaluate
# =========================
loss,acc=model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# =========================
# 13. Classification report
# =========================
y_true=np.concatenate(
    [y for x,y in test_ds],
    axis=0
)

preds=model.predict(test_ds)
y_pred=np.argmax(preds,axis=1)

print(
classification_report(
    y_true,
    y_pred
)
)

print(
confusion_matrix(
    y_true,
    y_pred
)
)

In [ ]:
# =========================
# 14. Save final model
# =========================
model.save(
    "models/mobilenetv3_trash_classifier.keras"
)